# 02_04: Image Segmentation

**Objective:** Learn to perform semantic, instance, and panoptic image segmentation using Transformer-based models.

**Prerequisites:** Basic Python, familiarity with HuggingFace pipelines (Notebook 00_03), image classification concepts (Notebook 02_01).

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|-------------|------------|------|---------|-------------------|-------|
| **Small (CPU)** | nvidia/segformer-b0-finetuned-ade-512-512 | ~14MB | 4GB RAM | Any CPU | Semantic segmentation |
| **Medium (GPU)** | facebook/mask2former-swin-base-ade-semantic | ~400MB | 8GB RAM | RTX 4080+ | Better accuracy |
| **Large (GPU)** | facebook/sam-vit-base | ~375MB | 12GB RAM | RTX 4080+ | Segment Anything Model |

## Expected Behaviors

### First Time Running
- **Model download**: ~14MB for SegFormer-B0, ~375-400MB for larger models
- Image downloads from URLs may take a few seconds

### What You'll See
- Semantic segmentation assigns a class label to every pixel in the image
- Different objects/regions get different colors in the segmentation mask
- SegFormer is very fast even on CPU; SAM requires more compute

### Common Observations
- Object boundaries may not be pixel-perfect, especially for small objects
- Indoor scenes with many overlapping objects are more challenging
- SAM can segment anything with point/box prompts but needs GPU for good speed

## Overview

**Image segmentation** classifies every pixel in an image, producing a detailed understanding of the scene. There are three main types:

| Type | Description | Example |
|------|-------------|--------|
| **Semantic** | Label every pixel with a class (no instance distinction) | All "car" pixels are the same class |
| **Instance** | Distinguish individual objects of the same class | Car #1 vs Car #2 vs Car #3 |
| **Panoptic** | Semantic + Instance combined | Stuff (sky, road) + Things (car #1, person #2) |

Segmentation is more fine-grained than object detection (which only draws bounding boxes). It's used in autonomous driving, medical imaging, satellite analysis, and photo editing.

## Setup and Installation

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import (
    pipeline,
    AutoImageProcessor,
    AutoModelForSemanticSegmentation,
    SegformerForSemanticSegmentation,
)
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
from io import BytesIO

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Model Selection

In [ ]:
# Option 1: Small Model (CPU-friendly, recommended for beginners)
MODEL_NAME = 'nvidia/segformer-b0-finetuned-ade-512-512'  # ~14MB, 150 classes

# Option 2: Medium Model (better accuracy, GPU recommended)
# MODEL_NAME = 'nvidia/segformer-b2-finetuned-ade-512-512'  # ~100MB

# Option 3: Large Model (state-of-the-art)
# MODEL_NAME = 'facebook/mask2former-swin-base-ade-semantic'  # ~400MB

## Helper Functions

Let's create utilities for loading images and visualizing segmentation results.

In [ ]:
def load_image_from_url(url: str, max_size: int = 512) -> Image.Image:
    """Load and resize an image from a URL.

    Args:
        url: URL of the image to load.
        max_size: Maximum dimension (width or height).

    Returns:
        PIL Image object.
    """
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    image = Image.open(BytesIO(response.content)).convert('RGB')
    
    # Resize if too large
    if max(image.size) > max_size:
        ratio = max_size / max(image.size)
        new_size = (int(image.size[0] * ratio), int(image.size[1] * ratio))
        image = image.resize(new_size, Image.LANCZOS)
    
    return image


# Generate a colormap for visualization
def create_colormap(num_classes: int) -> np.ndarray:
    """Create a colormap for segmentation visualization.

    Args:
        num_classes: Number of segmentation classes.

    Returns:
        NumPy array of shape (num_classes, 3) with RGB values.
    """
    np.random.seed(42)  # Fixed seed for consistent colors
    colors = np.random.randint(0, 255, size=(num_classes, 3), dtype=np.uint8)
    colors[0] = [0, 0, 0]  # Background is black
    return colors


print('Helper functions loaded.')

## Method 1: Pipeline API

The image segmentation pipeline handles preprocessing, model inference, and post-processing. It returns a list of segments, each with a label, score, and pixel mask.

In [ ]:
# Create segmentation pipeline
segmenter = pipeline(
    'image-segmentation',
    model=MODEL_NAME,
    device=device,
)

# Load a test image
try:
    image_url = 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/segmentation_input.jpg'
    test_image = load_image_from_url(image_url)
    print(f'Image loaded: {test_image.size}')
except Exception as error:
    print(f'Could not load image from URL: {error}')
    print('Creating a synthetic test image...')
    test_image = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))

# Run segmentation
segments = segmenter(test_image)

print(f'\nFound {len(segments)} segments:')
segment_df = pd.DataFrame([{
    'Label': s['label'],
    'Score': f'{s["score"]:.4f}' if s.get('score') else 'N/A',
    'Mask Shape': str(np.array(s['mask']).shape) if s.get('mask') else 'N/A',
} for s in segments[:15]])
segment_df

Each segment comes with a binary mask that indicates which pixels belong to that class. Let's visualize the original image alongside the segmentation result.

In [ ]:
def visualize_segmentation(
    image: Image.Image,
    segments: list[dict],
    max_segments: int = 10,
) -> None:
    """Visualize segmentation results overlaid on the original image.

    Args:
        image: Original PIL image.
        segments: List of segment dictionaries from the pipeline.
        max_segments: Maximum number of segments to display.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Original image
    axes[0].imshow(image)
    axes[0].set_title('Original Image', fontsize=13)
    axes[0].axis('off')
    
    # Segmentation overlay
    image_array = np.array(image)
    overlay = np.zeros_like(image_array)
    colors = create_colormap(max_segments + 1)
    
    legend_patches = []
    for i, segment in enumerate(segments[:max_segments]):
        if segment.get('mask') is None:
            continue
        mask = np.array(segment['mask'])
        if mask.ndim == 3:
            mask = mask[:, :, 0] > 0
        elif mask.ndim == 2:
            mask = mask > 0
        
        # Resize mask to match image if needed
        if mask.shape != image_array.shape[:2]:
            from PIL import Image as PILImage
            mask_img = PILImage.fromarray(mask.astype(np.uint8) * 255)
            mask_img = mask_img.resize(image.size, PILImage.NEAREST)
            mask = np.array(mask_img) > 0
        
        color = colors[i + 1]
        overlay[mask] = color
        legend_patches.append(
            mpatches.Patch(color=color / 255.0, label=segment['label'])
        )
    
    # Blend original with overlay
    blended = (image_array * 0.5 + overlay * 0.5).astype(np.uint8)
    axes[1].imshow(blended)
    axes[1].set_title('Segmentation Overlay', fontsize=13)
    axes[1].axis('off')
    if legend_patches:
        axes[1].legend(
            handles=legend_patches, loc='upper right',
            fontsize=8, framealpha=0.8,
        )
    
    plt.tight_layout()
    plt.show()


visualize_segmentation(test_image, segments)

## Method 2: Manual Model Loading

For more control, we can load the SegFormer model directly and process the raw logits. This allows us to inspect per-pixel class probabilities and apply custom post-processing.

In [ ]:
# Load model and processor
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = SegformerForSemanticSegmentation.from_pretrained(MODEL_NAME).to(device)
model.eval()

# Get class labels
id2label = model.config.id2label
num_classes = len(id2label)
print(f'Model: {MODEL_NAME}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Number of classes: {num_classes}')
print(f'\nSample classes: {[id2label[i] for i in range(min(10, num_classes))]}')

In [ ]:
def segment_image_manual(
    image: Image.Image,
    processor: transformers.ImageProcessingMixin,
    model: torch.nn.Module,
    id2label: dict[int, str],
) -> tuple[np.ndarray, dict[int, str]]:
    """Perform semantic segmentation with manual model inference.

    Args:
        image: Input PIL image.
        processor: Image processor for the model.
        model: Segmentation model.
        id2label: Mapping from class IDs to class names.

    Returns:
        Tuple of (segmentation map, mapping of present class IDs to names).
    """
    inputs = processor(images=image, return_tensors='pt').to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Upsample logits to original image size
    logits = outputs.logits
    upsampled = torch.nn.functional.interpolate(
        logits,
        size=image.size[::-1],  # (height, width)
        mode='bilinear',
        align_corners=False,
    )
    
    # Get per-pixel class predictions
    seg_map = upsampled.argmax(dim=1).squeeze().cpu().numpy()
    
    # Find which classes are present
    present_classes = {int(c): id2label[int(c)] for c in np.unique(seg_map)}
    
    return seg_map, present_classes


seg_map, present_classes = segment_image_manual(test_image, processor, model, id2label)

print(f'Segmentation map shape: {seg_map.shape}')
print(f'\nClasses found in image ({len(present_classes)}):')
for class_id, class_name in present_classes.items():
    pixel_count = (seg_map == class_id).sum()
    percentage = pixel_count / seg_map.size * 100
    print(f'  {class_name:20s} ({class_id:3d}): {percentage:5.1f}% of pixels')

In [ ]:
# Visualize the manual segmentation with per-class detail
def plot_segmentation_map(
    image: Image.Image,
    seg_map: np.ndarray,
    present_classes: dict[int, str],
) -> None:
    """Plot detailed segmentation map with class legend.

    Args:
        image: Original PIL image.
        seg_map: Integer array with per-pixel class IDs.
        present_classes: Dict mapping class IDs to names.
    """
    colors = create_colormap(max(present_classes.keys()) + 2)
    
    # Create colored segmentation map
    colored_map = colors[seg_map]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].imshow(image)
    axes[0].set_title('Original', fontsize=13)
    axes[0].axis('off')
    
    axes[1].imshow(colored_map)
    axes[1].set_title('Segmentation Map', fontsize=13)
    axes[1].axis('off')
    
    # Blended overlay
    image_array = np.array(image)
    blended = (image_array * 0.6 + colored_map * 0.4).astype(np.uint8)
    axes[2].imshow(blended)
    axes[2].set_title('Overlay', fontsize=13)
    axes[2].axis('off')
    
    # Legend
    patches = [
        mpatches.Patch(color=colors[cid] / 255.0, label=name)
        for cid, name in sorted(present_classes.items())
    ]
    fig.legend(
        handles=patches, loc='lower center',
        ncol=min(5, len(patches)), fontsize=8,
        bbox_to_anchor=(0.5, -0.05),
    )
    
    plt.tight_layout()
    plt.show()


plot_segmentation_map(test_image, seg_map, present_classes)

## Practical Applications

Image segmentation is used in autonomous driving (road/lane detection), medical imaging (tumor segmentation), satellite imagery (land use classification), and photo editing (background removal). Let's explore a practical scenario.

In [ ]:
# Application: Scene composition analysis
def analyze_scene_composition(
    image: Image.Image,
    processor: transformers.ImageProcessingMixin,
    model: torch.nn.Module,
    id2label: dict[int, str],
) -> pd.DataFrame:
    """Analyze the composition of a scene by class percentages.

    Args:
        image: Input PIL image.
        processor: Image processor.
        model: Segmentation model.
        id2label: Class ID to name mapping.

    Returns:
        DataFrame with class coverage percentages.
    """
    seg_map, present_classes = segment_image_manual(image, processor, model, id2label)
    total_pixels = seg_map.size
    
    rows: list[dict[str, str]] = []
    for class_id, class_name in sorted(
        present_classes.items(),
        key=lambda x: (seg_map == x[0]).sum(),
        reverse=True,
    ):
        count = int((seg_map == class_id).sum())
        rows.append({
            'Class': class_name,
            'Pixels': f'{count:,}',
            'Coverage': f'{count / total_pixels:.1%}',
        })
    
    rows.append({
        'Class': 'TOTAL',
        'Pixels': f'{total_pixels:,}',
        'Coverage': '100.0%',
    })
    
    return pd.DataFrame(rows)


print('=== Scene Composition Analysis ===')
analyze_scene_composition(test_image, processor, model, id2label)

## Performance Benchmarking

Segmentation speed depends on image resolution and model size. Let's measure how SegFormer performs on different input sizes.

In [ ]:
def benchmark_segmentation(
    image: Image.Image,
    sizes: list[int],
    processor: transformers.ImageProcessingMixin,
    model: torch.nn.Module,
    num_runs: int = 5,
) -> pd.DataFrame:
    """Benchmark segmentation speed at different resolutions.

    Args:
        image: Input PIL image.
        sizes: List of target sizes (max dimension).
        processor: Image processor.
        model: Segmentation model.
        num_runs: Number of runs for averaging.

    Returns:
        DataFrame with timing results per resolution.
    """
    results: list[dict[str, str]] = []
    
    for size in sizes:
        ratio = size / max(image.size)
        resized = image.resize(
            (int(image.size[0] * ratio), int(image.size[1] * ratio)),
            Image.LANCZOS,
        )
        
        inputs = processor(images=resized, return_tensors='pt').to(device)
        
        # Warmup
        with torch.no_grad():
            model(**inputs)
        
        times: list[float] = []
        for _ in range(num_runs):
            start = time.perf_counter()
            with torch.no_grad():
                model(**inputs)
            times.append(time.perf_counter() - start)
        
        results.append({
            'Resolution': f'{resized.size[0]}x{resized.size[1]}',
            'Pixels': f'{resized.size[0] * resized.size[1]:,}',
            'Mean Latency (ms)': f'{np.mean(times) * 1000:.1f}',
            'Std (ms)': f'{np.std(times) * 1000:.1f}',
            'Device': str(device),
        })
    
    return pd.DataFrame(results)


print('=== Segmentation Speed vs Resolution ===')
benchmark_segmentation(test_image, [256, 384, 512], processor, model)

## Exercises

1. **Compare models**: Load both `segformer-b0` and `segformer-b2` (if you have the RAM). Compare their segmentation quality and speed on the same image.

2. **Background removal**: Use segmentation to create a mask for the main subject, then replace the background with a solid color or another image.

3. **Class-specific extraction**: Extract only the pixels belonging to a specific class (e.g., "person" or "car") and display them isolated from the rest of the image.

4. **Custom image**: Run segmentation on your own photos. Try indoor scenes, outdoor landscapes, and street views to see how the model handles different environments.

## Key Takeaways

- **Image segmentation** classifies every pixel, providing much finer detail than object detection (bounding boxes)
- **Three types**: Semantic (class per pixel), Instance (distinguish objects), Panoptic (both)
- **SegFormer** is a lightweight and efficient model — SegFormer-B0 is only ~14MB and runs well on CPU
- Segmentation quality depends on image resolution, scene complexity, and model size
- **Applications** include autonomous driving, medical imaging, satellite analysis, and photo editing

## Next Steps & Resources

**Next section**: [03_01 — Speech Recognition](../03_audio/03_01_audio_speech_recognition.ipynb) — move from vision to audio processing.

**Resources:**
- [HuggingFace Segmentation Guide](https://huggingface.co/docs/transformers/tasks/semantic_segmentation)
- [SegFormer Paper](https://arxiv.org/abs/2105.15203) — Simple and Efficient Design for Semantic Segmentation
- [SAM (Segment Anything)](https://segment-anything.com/) — Meta's universal segmentation model
- [Mask2Former Paper](https://arxiv.org/abs/2112.01527) — Unified segmentation architecture
- [Segmentation Models on Hub](https://huggingface.co/models?pipeline_tag=image-segmentation)